# True Out-of-Sample: 2026 Live Test (Jan 1 – May 15)

**This notebook answers the only question that really matters for a trading strategy:**

> *Does the model make money on data that didn't exist when it was built?*

---

### What makes this genuinely out-of-sample

| Period | Role | Status |
|---|---|---|
| 2015–2019 | Training | Model coefficients fixed. Never touched again. |
| 2020–2021 | Validation | Thresholds τ chosen. Never touched again. |
| 2022–2025 | Test (out-of-sample window) | Evaluated once. |
| **2026 Jan 1 – May 15** | **This notebook** | **Never seen by any part of the pipeline.** |

NYISO prices for January–May 2026 are genuinely post-deployment data — none of  
the model fitting, threshold tuning, or earlier backtests had access to them.

**No retraining. No threshold re-tuning. No look at results before committing.**  
We apply the model exactly as it was trained years ago and see what happens.

In [ ]:
import warnings; warnings.filterwarnings('ignore')

import os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import joblib

# Project root
PROJECT = Path(os.getcwd()).parent
sys.path.insert(0, str(PROJECT))

DATA    = PROJECT / 'data'
MODELS  = PROJECT / 'models'
RESULTS = PROJECT / 'results'
FEATS   = DATA / 'features'

plt.rcParams.update({
    'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3,
    'axes.spines.top': False, 'axes.spines.right': False,
})

# Paper constants (unchanged from training)
GAMMA_POS = 5.0    # $/MWh  positive DART spike threshold (DEC signal)
GAMMA_NEG = 30.0   # $/MWh  negative DART spike threshold (INC signal)
ZONES = ['CAPITL','CENTRL','DUNWOD','GENESE','HUDVL',
         'LONGIL','MHKVL','MILLWD','NORTH','NYC','WEST']

# Cutoff: only use data through May 15 (RT for May 16 is still streaming)
LIVE_END = pd.Timestamp('2026-05-15 23:00:00', tz='America/New_York')

print('Project root:', PROJECT)
print('Models dir  :', MODELS)
print('Live data cutoff:', LIVE_END)

---
## Step 1 — Load the 2026 panel

The panel was built by extending the download to 2026 and re-running the build pipeline  
with [2025, 2026] so that December 2025 data is available for January 2026 lag features.

The panel contains: DA LMP, RT LMP, DART, DA load forecast, actual load — all joined  
on (interval_start_local, zone). Exactly the same format as the historical panel.

In [ ]:
# Load the combined 2025+2026 panel
panel_full = pd.read_parquet(DATA / 'processed' / 'panel_live_2026.parquet')

# Filter to 2026 data up to the cutoff
is_2026 = (panel_full['interval_start_local'].dt.year == 2026) & \
          (panel_full['interval_start_local'] <= LIVE_END)
panel_2026 = panel_full[is_2026].copy()

print(f'2025+2026 panel rows: {len(panel_full):,}')
print(f'2026 rows (through May 15): {len(panel_2026):,}')
print(f'2026 hours: {panel_2026["interval_start_local"].nunique():,}')
print(f'Zones: {sorted(panel_2026["zone"].unique())}')
print(f'Date range: {panel_2026["interval_start_local"].min().date()} '
      f'→ {panel_2026["interval_start_local"].max().date()}')
print()
print('2026 YTD DART statistics (all zones):')
print(panel_2026['dart'].describe().round(2).to_string())

In [ ]:
# ── 2026 DART overview: distribution and zone comparison ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
dart_clip = panel_2026['dart'].clip(-100, 100)
ax.hist(dart_clip, bins=150, color='steelblue', alpha=0.75, edgecolor='none')
ax.axvline( GAMMA_POS, color='green', lw=1.5, label=f'γ_pos = +${GAMMA_POS:.0f}')
ax.axvline(-GAMMA_NEG, color='red',   lw=1.5, label=f'γ_neg = -${GAMMA_NEG:.0f}')
pos_rate = (panel_2026['dart'] >= GAMMA_POS).mean()
neg_rate = (panel_2026['dart'] <= -GAMMA_NEG).mean()
ax.set_title(f'2026 DART distribution (clipped ±$100)\n'
             f'Pos spikes: {pos_rate:.1%}  |  Neg spikes: {neg_rate:.1%}')
ax.set_xlabel('DART ($/MWh)'); ax.set_ylabel('Hours'); ax.legend()

ax = axes[1]
zone_means = panel_2026.groupby('zone')['dart'].mean().reindex(ZONES)
colors = ['#2e7d32' if v >= 0 else '#c62828' for v in zone_means]
ax.bar(ZONES, zone_means, color=colors, alpha=0.8)
ax.axhline(0, color='black', lw=0.5)
ax.axhline( GAMMA_POS, color='green', lw=1, ls='--', alpha=0.6, label='γ_pos')
ax.axhline(-GAMMA_NEG, color='red',   lw=1, ls='--', alpha=0.6, label='γ_neg')
ax.set_title('Mean DART by zone — 2026 YTD\n(green = DA > RT on avg, red = RT > DA on avg)')
ax.set_xlabel('Zone'); ax.set_ylabel('Mean DART ($/MWh)')
ax.tick_params(axis='x', rotation=45); ax.legend(fontsize=8)

plt.tight_layout(); plt.show()

print(f'Spike prevalence vs 2015-2025 historical:')
print(f'  Positive DART ≥ +${GAMMA_POS}:  2026={pos_rate:.1%}  vs historical ~30%')
print(f'  Negative DART ≤ -${GAMMA_NEG}: 2026={neg_rate:.1%}  vs historical ~3%')

---
## Step 2 — Build 2026 feature matrix

We use the same feature engineering pipeline as training, applied to 2026 data.  
The December 2025 tail in the panel provides the lag-24 and lag-48 values for early January 2026.

**Nothing is re-estimated here.** The feature definitions are mechanical functions  
of the data. The StandardScaler inside each model (fitted on 2015–2019) will handle  
the normalization when predictions are made.

In [ ]:
from nyiso_dart.features.build import build_features

print('Building features for 2025+2026 panel...')
artefacts = build_features(panel_full)   # uses both years so Dec 2025 feeds Jan 2026 lags

X_all  = artefacts['X_naive']   # literal t-24h lag (standard time-series convention)
y_all  = artefacts['y']

# Filter to 2026 up to cutoff
mask_2026 = (X_all.index.year == 2026) & (X_all.index <= LIVE_END)
X_2026 = X_all[mask_2026]
y_2026 = y_all[mask_2026]

print(f'\nFeature matrix: {X_2026.shape[0]:,} hours × {X_2026.shape[1]} features')
print(f'Label matrix:   {y_2026.shape[0]:,} hours × {y_2026.shape[1]} labels')
print(f'NaN rows in X:  {X_2026.isna().any(axis=1).sum():,}  (lag features missing for very first days)')
print()
print('2026 YTD label prevalence:')
print(f"{'Zone':8s} {'Pos spikes':>12s} {'Neg spikes':>12s}")
for z in ZONES:
    p = float(y_2026[f'{z}_pos'].mean())
    n = float(y_2026[f'{z}_neg'].mean())
    print(f'{z:8s} {p:>12.1%} {n:>12.1%}')

---
## Step 3 — Apply the 2015–2019 trained models

Load each of the 22 fitted sklearn Pipelines from disk.  
Each Pipeline contains a StandardScaler fitted on 2015–2019 data followed by  
a LogisticRegression with coefficients frozen since training.

**The scaler handles normalization automatically** — 2026 features are scaled  
using the same mean/std the model saw during training. If 2026 values are  
out-of-distribution, that will show up as unusually high or low predicted probabilities.

In [ ]:
print('Loading 22 trained models and generating predictions...')
preds_2026 = pd.DataFrame(index=X_2026.index, dtype='float64')
model_stats = []

valid_X = X_2026.notna().all(axis=1)
X_valid = X_2026[valid_X]

for zone in ZONES:
    for side in ('pos', 'neg'):
        label = f'{zone}_{side}'
        mpath = MODELS / 'naive' / f'{zone}_{side}.joblib'
        pipe  = joblib.load(mpath)

        # Predict on rows with complete features
        proba = pd.Series(np.nan, index=X_2026.index)
        proba[valid_X] = pipe.predict_proba(X_valid.values)[:, 1]
        preds_2026[label] = proba

        model_stats.append({
            'label': label,
            'mean_proba': float(proba.dropna().mean()),
            'max_proba': float(proba.dropna().max()),
        })

print(f'Done — predictions generated for {len(preds_2026):,} hours.')
print()
print(f'Predicted probability summary (mean across 2026 hours):')
ms = pd.DataFrame(model_stats).sort_values('mean_proba', ascending=False)
print(f"{'Label':20s} {'Mean prob':>10s} {'Max prob':>10s}")
print('-' * 44)
for _, r in ms.head(12).iterrows():
    print(f"{r.label:20s} {r.mean_proba:>10.4f} {r.max_proba:>10.4f}")

---
## Step 4 — Apply validation-tuned thresholds → trade signals

Use the τ values chosen on the **2020–2021 validation set**.  
These have not been touched since. Only eligible (zone, side) pairs trade.

In [ ]:
# Load thresholds (frozen since validation tuning)
thr = pd.read_parquet(MODELS / 'thresholds_naive.parquet')
elig = thr[thr['eligible']].copy()

# DART series for 2026 (wide format)
dart_wide = panel_full.pivot(
    index='interval_start_local', columns='zone', values='dart'
)[ZONES]
dart_2026 = dart_wide[(dart_wide.index.year == 2026) & (dart_wide.index <= LIVE_END)]

# Generate trades
trade_rows = []
for _, row in elig.iterrows():
    zone, side, tau = row['zone'], row['side'], float(row['best_tau'])
    label = f'{zone}_{side}'

    p = preds_2026[label]
    d = dart_2026[zone]
    ok = p.notna() & d.notna() & (p >= tau)
    if not ok.any(): continue

    payoff = d[ok] if side == 'pos' else -d[ok]
    for t, pval, dart_val, pay in zip(ok[ok].index, p[ok], d[ok], payoff):
        trade_rows.append({
            'interval_start_local': t, 'zone': zone, 'side': side,
            'proba': float(pval), 'dart': float(dart_val), 'payoff': float(pay),
        })

trades = pd.DataFrame(trade_rows).sort_values('interval_start_local').reset_index(drop=True)
trades['month'] = trades['interval_start_local'].dt.month
trades['week']  = trades['interval_start_local'].dt.isocalendar().week.astype(int)
trades['correct'] = np.where(
    trades['side'] == 'pos',
    trades['dart'] >= GAMMA_POS,
    trades['dart'] <= -GAMMA_NEG
)

print(f'Trades executed Jan 1 – May 15, 2026: {len(trades):,}')
print(f'Eligible (zone, side) pairs: {elig["eligible"].sum()}')
print()
print(f'Trades by side:')
print(trades.groupby('side')[['payoff']].agg(['count','sum','mean']).round(2).to_string())
print()
print(f'Total P&L: ${trades["payoff"].sum():,.2f}')
print(f'Win rate:  {(trades["payoff"] > 0).mean():.1%}')

In [ ]:
# ── Cumulative P&L — 2026 YTD ──────────────────────────────────────────────────
hourly_idx = dart_2026.index
hourly_pnl = pd.Series(0.0, index=hourly_idx)
for t, pay in zip(trades['interval_start_local'], trades['payoff']):
    if t in hourly_pnl.index:
        hourly_pnl[t] += pay
cum_pnl = hourly_pnl.cumsum()

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

ax = axes[0]
ax.plot(cum_pnl.index, cum_pnl.values, lw=1.8, color='#1f4e79')
ax.fill_between(cum_pnl.index, 0, cum_pnl.values,
                where=cum_pnl.values >= 0, alpha=0.08, color='#1f4e79')
ax.fill_between(cum_pnl.index, 0, cum_pnl.values,
                where=cum_pnl.values < 0,  alpha=0.15, color='crimson')
ax.axhline(0, color='grey', lw=0.5)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.set_title('Cumulative P&L — Jan 1 to May 15, 2026  (unit size: 1 MWh/trade)\n'
             'Models trained 2015-2019, thresholds tuned 2020-2021 — zero re-tuning on 2026')
ax.set_ylabel('USD')

# DEC vs INC breakdown
pos_hourly = pd.Series(0.0, index=hourly_idx)
neg_hourly = pd.Series(0.0, index=hourly_idx)
for _, r in trades.iterrows():
    if r['interval_start_local'] in hourly_idx:
        if r['side'] == 'pos': pos_hourly[r['interval_start_local']] += r['payoff']
        else:                  neg_hourly[r['interval_start_local']] += r['payoff']

ax = axes[1]
ax.plot(cum_pnl.index, pos_hourly.cumsum(), lw=1.2, color='#2e7d32', label='DEC (pos side)')
ax.plot(cum_pnl.index, neg_hourly.cumsum(), lw=1.2, color='#c62828', label='INC (neg side)')
ax.axhline(0, color='grey', lw=0.5)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.set_ylabel('USD'); ax.set_xlabel('Hour')
ax.set_title('DEC vs INC side attribution')
ax.legend()

plt.tight_layout(); plt.show()
print(f'Final cumulative P&L (May 15, 2026): ${cum_pnl.iloc[-1]:,.2f}')

In [ ]:
# ── Monthly P&L breakdown ──────────────────────────────────────────────────────
month_names = {1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May'}
monthly = trades.groupby('month')['payoff'].agg(['sum','count','mean'])
monthly.index = monthly.index.map(month_names)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
colors = ['#2e7d32' if v >= 0 else '#c62828' for v in monthly['sum']]
ax.bar(monthly.index, monthly['sum'], color=colors, alpha=0.8)
ax.axhline(0, color='black', lw=0.5)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.set_title('Monthly P&L — 2026 YTD'); ax.set_xlabel('Month'); ax.set_ylabel('USD')

ax = axes[1]
ax.bar(monthly.index, monthly['count'], color='steelblue', alpha=0.8)
ax.set_title('Monthly trade count — 2026 YTD')
ax.set_xlabel('Month'); ax.set_ylabel('Number of trades')

plt.tight_layout(); plt.show()

print('Monthly breakdown:')
print(f"{'Month':6s} {'P&L':>12s} {'Trades':>8s} {'Avg/trade':>10s}")
print('-' * 40)
for m, r in monthly.iterrows():
    print(f'{m:6s} ${r["sum"]:>10,.2f} {r["count"]:>8.0f} ${r["mean"]:>8,.2f}')
print()
print(f'Total: ${monthly["sum"].sum():,.2f}  across {int(monthly["count"].sum())} trades')

In [ ]:
# ── Per-zone attribution ────────────────────────────────────────────────────────
zone_attr = (trades
    .groupby(['zone','side'])
    .agg(n_trades=('payoff','size'), total_pnl=('payoff','sum'),
         avg_pnl=('payoff','mean'), precision=('correct','mean'))
    .sort_values('total_pnl', ascending=False)
    .reset_index())

fig, ax = plt.subplots(figsize=(9, 6))
labels = zone_attr['zone'] + '/' + zone_attr['side']
colors = ['#2e7d32' if s == 'pos' else '#c62828' for s in zone_attr['side']]
ax.barh(labels[::-1], zone_attr['total_pnl'][::-1], color=colors[::-1], alpha=0.8)
ax.axvline(0, color='black', lw=0.5)
ax.set_xlabel('Total P&L (USD)')
ax.set_title('Per-(zone, side) attribution — 2026 YTD\nGreen=DEC, Red=INC')
plt.tight_layout(); plt.show()

print('Full attribution:')
print(f"{'Zone':8s} {'Side':5s} {'Trades':>8s} {'Precision':>10s} {'Total P&L':>12s} {'Avg/trade':>10s}")
print('-' * 60)
for _, r in zone_attr.iterrows():
    print(f"{r.zone:8s} {r.side:5s} {r.n_trades:>8.0f} {r.precision:>10.1%} "
          f"${r.total_pnl:>10,.2f} ${r.avg_pnl:>8,.2f}")

---
## Step 5 — Compare to prior years at same point in calendar

To contextualize 2026, we compare cumulative P&L **through May 15** across all  
test years: 2022, 2023, 2024, 2025, and now 2026.

This answers: *Is 2026 running ahead of, behind, or in line with historical pacing?*

In [ ]:
# Load historical test trades (2022-2025)
hist = pd.read_parquet(RESULTS / 'naive' / 'trades.parquet')

# For each year, compute cumulative P&L through May 15 (day 135)
# using day-of-year as the x-axis for alignment
fig, ax = plt.subplots(figsize=(13, 6))

colors_yr = {2022:'#e57373', 2023:'#ff9800', 2024:'#4caf50',
             2025:'#1565c0', 2026:'#6a1b9a'}

for year in [2022, 2023, 2024, 2025]:
    y_trades = hist[hist['year'] == year].copy()
    if y_trades.empty: continue

    # Index by day-of-year
    y_trades['doy'] = y_trades['interval_start_local'].dt.day_of_year
    cum_by_doy = y_trades.groupby('doy')['payoff'].sum().cumsum()

    # Cap to May 15 = day 135 (non-leap year)
    cap_doy = 135
    cum_cap = cum_by_doy[cum_by_doy.index <= cap_doy]
    final = float(cum_by_doy[cum_by_doy.index <= cap_doy].iloc[-1]) if len(cum_cap) else 0
    ax.plot(cum_cap.index, cum_cap.values,
            lw=1.4, color=colors_yr[year],
            label=f'{year}  (through May 15: ${final:,.0f})', alpha=0.8)

# 2026
trades['doy'] = trades['interval_start_local'].dt.day_of_year
cum_2026_doy = trades.groupby('doy')['payoff'].sum().cumsum()
final_2026 = float(cum_2026_doy.iloc[-1])
ax.plot(cum_2026_doy.index, cum_2026_doy.values,
        lw=2.4, color=colors_yr[2026],
        label=f'2026  (through May 15: ${final_2026:,.0f})', zorder=5)

ax.axhline(0, color='grey', lw=0.5)
ax.axvline(135, color='grey', lw=0.5, ls='--', alpha=0.5, label='May 15 (day 135)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.set_xlabel('Day of year (Jan 1 = day 1)')
ax.set_ylabel('Cumulative P&L (USD, unit size)')
ax.set_title('Jan 1 → May 15 cumulative P&L comparison: 2022-2026\n'
             '2026 (purple) uses zero re-tuning — pure out-of-sample')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout(); plt.show()

print('Through-May-15 P&L by year:')
for year in [2022, 2023, 2024, 2025]:
    y_trades = hist[hist['year'] == year]
    y_trades = y_trades.assign(doy=y_trades['interval_start_local'].dt.day_of_year)
    pnl = y_trades[y_trades['doy'] <= 135]['payoff'].sum()
    print(f'  {year}: ${pnl:>10,.2f}')
print(f'  2026: ${final_2026:>10,.2f}  ← TODAY')

In [ ]:
# ── Full 2022-2025 context: how does Jan-May typically look? ──────────────────
# Show full-year equity curves with Jan-May highlighted
cum_hist = pd.read_parquet(RESULTS / 'naive' / 'cumulative_pnl.parquet')
full_pnl = float(cum_hist['pnl_total'].iloc[-1])
jan_may_pnl = float(hist[hist['interval_start_local'].dt.month <= 5]['payoff'].sum())

print('2022-2025 full test period context:')
print(f'  Total P&L 2022-2025:         ${full_pnl:>12,.2f}')
print(f'  Jan-May P&L 2022-2025:       ${jan_may_pnl:>12,.2f}  ({jan_may_pnl/full_pnl:.1%} of total)')
print(f'  Jun-Dec P&L 2022-2025:       ${full_pnl-jan_may_pnl:>12,.2f}  ({(full_pnl-jan_may_pnl)/full_pnl:.1%} of total)')
print()
print('2026 YTD (Jan 1 – May 15):')
print(f'  P&L:    ${final_2026:>12,.2f}')
print(f'  Trades: {len(trades):>12,}')
print()
print('Context: summer months (Jun-Sep) have historically been the strongest')
print('period for INC alpha (peak load → LONGIL and NYC negative DART spikes).')
print('2026 summer performance is yet to come.')

In [ ]:
# ── Trading statistics ─────────────────────────────────────────────────────────
n = len(trades)
total_pnl = float(trades['payoff'].sum())
wins   = trades[trades['payoff'] > 0]['payoff']
losses = trades[trades['payoff'] < 0]['payoff']

daily_pnl = pd.Series(
    trades.groupby(trades['interval_start_local'].dt.date)['payoff'].sum()
)
# Annualize from 135 days to 365
ann_scale = 365 / 135
sharpe = (daily_pnl.mean() / daily_pnl.std() * np.sqrt(365)) if daily_pnl.std() > 0 else 0

# Drawdown
running_max = cum_pnl.cummax()
dd = (cum_pnl - running_max)
max_dd = float(dd.min())

profit_factor = (wins.sum() / abs(losses.sum())) if len(losses) else float('inf')
precision = float(trades['correct'].mean())

print('─' * 50)
print('  2026 YTD TRADING STATISTICS (Jan 1 – May 15)')
print('─' * 50)
print(f'  Trades executed         {n:>12,}')
print(f'  Total P&L               ${total_pnl:>11,.2f}')
print(f'  Annualized P&L (proj.)  ${total_pnl*ann_scale:>11,.2f}  (× {ann_scale:.1f}x scale)')
print(f'  Win rate                {(trades["payoff"]>0).mean():>12.1%}')
print(f'  Profit factor           {profit_factor:>12.2f}x')
print(f'  Avg win                 ${wins.mean():>11,.2f}')
print(f'  Avg loss                ${losses.mean():>11,.2f}')
print(f'  Best trade              ${trades["payoff"].max():>11,.2f}')
print(f'  Worst trade             ${trades["payoff"].min():>11,.2f}')
print(f'  Sharpe (daily, ann.)    {sharpe:>12.2f}')
print(f'  Max drawdown            ${max_dd:>11,.2f}')
print(f'  Precision (spike hit)   {precision:>12.1%}')
print('─' * 50)

In [ ]:
# ── Most profitable individual trades ─────────────────────────────────────────
print('Top 10 most profitable trades in 2026:')
top = trades.nlargest(10, 'payoff')[
    ['interval_start_local','zone','side','proba','dart','payoff','correct']
]
for _, r in top.iterrows():
    print(f"  {str(r.interval_start_local)[:22]}  {r.zone:6s} {r.side:3s}  "
          f"p={r.proba:.2f}  DART={r.dart:>8.2f}  payoff=${r.payoff:>8.2f}  "
          f"{'✓' if r.correct else '✗'}")

print()
print('Worst 5 trades:')
worst = trades.nsmallest(5, 'payoff')[
    ['interval_start_local','zone','side','proba','dart','payoff']
]
for _, r in worst.iterrows():
    print(f"  {str(r.interval_start_local)[:22]}  {r.zone:6s} {r.side:3s}  "
          f"p={r.proba:.2f}  DART={r.dart:>8.2f}  payoff=${r.payoff:>8.2f}")

In [ ]:
# ── DART regime check: is 2026 different from training distribution? ───────────
panel_hist = pd.read_parquet(DATA / 'processed' / 'panel.parquet')

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, zone in zip(axes, ['LONGIL', 'NYC', 'CAPITL']):
    h = panel_hist[panel_hist['zone'] == zone]['dart'].clip(-50, 50)
    l = panel_2026[panel_2026['zone'] == zone]['dart'].clip(-50, 50)
    bins = np.linspace(-50, 50, 60)
    ax.hist(h, bins=bins, density=True, alpha=0.5, color='steelblue',
            label=f'2015-2025 (μ={h.mean():.1f})')
    ax.hist(l, bins=bins, density=True, alpha=0.6, color='#6a1b9a',
            label=f'2026 YTD  (μ={l.mean():.1f})')
    ax.axvline( GAMMA_POS, color='green', lw=1, ls='--')
    ax.set_title(zone); ax.set_xlabel('DART ($/MWh)')
    ax.legend(fontsize=8)

plt.suptitle('DART distribution: 2015-2025 training period vs 2026 YTD\n'
             'A rightward shift means more positive DART (DEC-favorable environment)', y=1.03)
plt.tight_layout(); plt.show()

print('Distribution shift summary:')
for zone in ZONES:
    h_mean = panel_hist[panel_hist['zone']==zone]['dart'].mean()
    l_mean = panel_2026[panel_2026['zone']==zone]['dart'].mean()
    shift = l_mean - h_mean
    arrow = '↑' if shift > 0 else '↓'
    print(f'  {zone:8s}  hist={h_mean:>6.2f}  2026={l_mean:>6.2f}  shift={shift:>+6.2f} {arrow}')

---
## Summary

### What we just did

Applied a model trained on 2015–2019 data and threshold-tuned on 2020–2021 to
January 1 – May 15, 2026 — genuinely never-before-seen data.

### Takeaways

| Question | Finding |
|---|---|
| Does the model still fire signals on new data? | Yes — trades execute in all months |
| Is the 2026 DART environment similar to training? | Distribution has shifted (check the bar charts above) |
| How does Jan-May 2026 compare to prior Jan-May periods? | See the year-over-year comparison chart |
| Is there a signal asymmetry? | Check which side (INC vs DEC) is driving more P&L |

### Critical context

January–May is historically the **weakest** period for this strategy.  
The summer 2025 spike episode drove roughly 49% of the 2022-2025 test P&L in a handful  
of days. The real test of whether 2026 continues the pattern comes with summer peak  
season (Jun–Aug).

### What the distribution shift means

If 2026 shows a rightward DART shift (higher positive DART on average), that means:
- DA has been over-clearing relative to RT in 2026
- DEC trades (profit when DA > RT) face a **tailwind** — the drift is in their favor
- INC trades (profit when RT > DA) face a **headwind** — negative DART spikes are rarer

A model trained on 2015–2019 patterns may under-predict DEC opportunities in this environment
(calibrated for a more neutral DART distribution). This is **regime drift** — a real risk in
any historical model applied to live markets.